In [2]:
!python run_embedding_matching.py --backend openai

python3: can't open file '/content/run_embedding_matching.py': [Errno 2] No such file or directory


In [20]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer


# ============================================================
# CONFIG
# ============================================================

TAXONOMY_PATH = "saudi_skills_taxonomy_v1_final.csv"
GROUND_TRUTH_PATH = "annotation_sheet_full_Lama.csv"

RESULTS_PATH = "embedding_results.csv"
EVAL_PATH = "embedding_evaluation.csv"

MODEL_NAME = "intfloat/multilingual-e5-base"

TOP_K = 5


# ============================================================
# EXACT COLUMN NAMES FROM YOUR CSV
# ============================================================

FILE_COLUMN = "file_name"

TAG_COLUMN = (
    "predicted_tags "
    "(3-8, semicolon-separated, specific not broad)"
)

COMPETENCY_COLUMN = (
    "proposed_competencies "
    "(1-3 skill names, COPIED EXACTLY from valid_taxonomy_skill_reference.csv)"
)


# ============================================================
# LOAD DATA
# ============================================================

taxonomy = pd.read_csv(TAXONOMY_PATH)
gt = pd.read_csv(GROUND_TRUTH_PATH)

print("Taxonomy rows:", len(taxonomy))
print("Ground-truth rows:", len(gt))


# ============================================================
# CHECK COLUMNS
# ============================================================

required_taxonomy = [
    "skill_name_en",
    "description_en",
    "subsector_en",
    "related_job_families_en"
]

for col in required_taxonomy:
    if col not in taxonomy.columns:
        raise ValueError(
            f"Missing taxonomy column: {col}"
        )

for col in [
    FILE_COLUMN,
    TAG_COLUMN,
    COMPETENCY_COLUMN
]:
    if col not in gt.columns:
        raise ValueError(
            f"Missing ground-truth column: {col}"
        )


# ============================================================
# CLEAN TAXONOMY
# ============================================================

for col in required_taxonomy:
    taxonomy[col] = (
        taxonomy[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

taxonomy = taxonomy.drop_duplicates(
    subset=["skill_name_en"]
).reset_index(drop=True)


# ============================================================
# BUILD TAXONOMY TEXT
# ============================================================

# Competency name only
taxonomy["skill_text"] = (
    "passage: "
    + taxonomy["skill_name_en"]
)

# Competency + description + context
taxonomy["full_text"] = (
    "passage: "
    + taxonomy["skill_name_en"]
    + ". "
    + taxonomy["description_en"]
    + ". Subsector: "
    + taxonomy["subsector_en"]
    + ". Related job families: "
    + taxonomy["related_job_families_en"]
)


# ============================================================
# LOAD MODEL
# ============================================================

print("\nLoading model...")

model = SentenceTransformer(MODEL_NAME)


# ============================================================
# EMBED TAXONOMY
# ============================================================

print("\nEmbedding competency names...")

skill_embeddings = model.encode(
    taxonomy["skill_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=32
)

print("\nEmbedding full taxonomy descriptions...")

full_embeddings = model.encode(
    taxonomy["full_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=32
)


# ============================================================
# HELPERS
# ============================================================

def split_values(value):

    if pd.isna(value):
        return []

    value = str(value).strip()

    if not value:
        return []

    return [
        x.strip()
        for x in value.split(";")
        if x.strip()
    ]


def normalize_text(text):

    return (
        str(text)
        .strip()
        .lower()
        .replace("–", "-")
        .replace("—", "-")
    )


# ============================================================
# PROCESS EACH EVALUATION FILE
# ============================================================

results = []

top1_correct = 0
top3_correct = 0
top5_correct = 0

reciprocal_ranks = []

total_matched = 0
total_predicted = 0
total_ground_truth = 0


for _, row in gt.iterrows():

    file_name = row[FILE_COLUMN]

    # --------------------------------------------------------
    # QUERY = CONTENT TAGS ONLY
    # --------------------------------------------------------

    tags = split_values(
        row[TAG_COLUMN]
    )

    query_text = (
        "query: Learning content covering "
        + "; ".join(tags)
    )

    # --------------------------------------------------------
    # EMBED QUERY
    # --------------------------------------------------------

    query_embedding = model.encode(
        query_text,
        normalize_embeddings=True
    )

    # --------------------------------------------------------
    # SIMILARITY 1:
    # Competency name
    # --------------------------------------------------------

    skill_scores = np.dot(
        skill_embeddings,
        query_embedding
    )

    # --------------------------------------------------------
    # SIMILARITY 2:
    # Full taxonomy information
    # --------------------------------------------------------

    full_scores = np.dot(
        full_embeddings,
        query_embedding
    )

    # --------------------------------------------------------
    # COMBINED SCORE
    # --------------------------------------------------------

    combined_scores = (
        0.60 * skill_scores
        + 0.40 * full_scores
    )

    # --------------------------------------------------------
    # TOP 5
    # --------------------------------------------------------

    top_indices = np.argsort(
        combined_scores
    )[::-1][:TOP_K]

    predicted_skills = [
        taxonomy.iloc[i]["skill_name_en"]
        for i in top_indices
    ]

    predicted_scores = [
        float(combined_scores[i])
        for i in top_indices
    ]

    # --------------------------------------------------------
    # GROUND TRUTH COMPETENCIES
    # --------------------------------------------------------

    ground_truth = split_values(
        row[COMPETENCY_COLUMN]
    )

    gt_norm = {
        normalize_text(x)
        for x in ground_truth
    }

    predicted_norm = [
        normalize_text(x)
        for x in predicted_skills
    ]

    # --------------------------------------------------------
    # MATCHING
    # --------------------------------------------------------

    matched = [
        x for x in predicted_skills
        if normalize_text(x) in gt_norm
    ]

    missed = [
        x for x in ground_truth
        if normalize_text(x)
        not in {
            normalize_text(m)
            for m in matched
        }
    ]

    false_positives = [
        x for x in predicted_skills
        if normalize_text(x) not in gt_norm
    ]

    # --------------------------------------------------------
    # TOP-1
    # --------------------------------------------------------

    top1_match = (
        len(predicted_norm) > 0
        and predicted_norm[0] in gt_norm
    )

    # --------------------------------------------------------
    # TOP-3
    # --------------------------------------------------------

    top3_match = any(
        x in gt_norm
        for x in predicted_norm[:3]
    )

    # --------------------------------------------------------
    # TOP-5
    # --------------------------------------------------------

    top5_match = any(
        x in gt_norm
        for x in predicted_norm[:5]
    )

    if top1_match:
        top1_correct += 1

    if top3_match:
        top3_correct += 1

    if top5_match:
        top5_correct += 1

    # --------------------------------------------------------
    # FIRST MATCH RANK / MRR
    # --------------------------------------------------------

    first_match_rank = None

    for rank, prediction in enumerate(
        predicted_norm,
        start=1
    ):

        if prediction in gt_norm:

            first_match_rank = rank

            reciprocal_ranks.append(
                1.0 / rank
            )

            break

    if first_match_rank is None:
        reciprocal_ranks.append(0.0)

    # --------------------------------------------------------
    # PRECISION / RECALL / F1
    # --------------------------------------------------------

    matched_count = len(matched)
    predicted_count = len(predicted_skills)
    ground_truth_count = len(ground_truth)

    precision = (
        matched_count / predicted_count
        if predicted_count > 0
        else 0
    )

    recall = (
        matched_count / ground_truth_count
        if ground_truth_count > 0
        else 0
    )

    if precision + recall > 0:

        f1 = (
            2 * precision * recall
            / (precision + recall)
        )

    else:
        f1 = 0

    total_matched += matched_count
    total_predicted += predicted_count
    total_ground_truth += ground_truth_count

    # --------------------------------------------------------
    # STORE
    # --------------------------------------------------------

    results.append({

        "file_name": file_name,

        "ground_truth_tags":
            "; ".join(tags),

        "ground_truth_competencies":
            "; ".join(ground_truth),

        "predicted_competencies":
            "; ".join(predicted_skills),

        "predicted_scores":
            "; ".join(
                f"{score:.4f}"
                for score in predicted_scores
            ),

        "matched_competencies":
            "; ".join(matched),

        "missed_competencies":
            "; ".join(missed),

        "false_positive_competencies":
            "; ".join(false_positives),

        "first_match_rank":
            first_match_rank,

        "top1_match":
            int(top1_match),

        "top3_match":
            int(top3_match),

        "top5_match":
            int(top5_match),

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1
    })


# ============================================================
# SAVE DETAILED RESULTS
# ============================================================

results_df = pd.DataFrame(results)

results_df.to_csv(
    RESULTS_PATH,
    index=False
)


# ============================================================
# OVERALL METRICS
# ============================================================

num_files = len(gt)

top1_accuracy = (
    top1_correct / num_files
)

top3_accuracy = (
    top3_correct / num_files
)

top5_accuracy = (
    top5_correct / num_files
)

mrr = np.mean(
    reciprocal_ranks
)

overall_precision = (
    total_matched / total_predicted
)

overall_recall = (
    total_matched / total_ground_truth
)

if overall_precision + overall_recall > 0:

    overall_f1 = (
        2
        * overall_precision
        * overall_recall
        / (
            overall_precision
            + overall_recall
        )
    )

else:
    overall_f1 = 0


# ============================================================
# SAVE SUMMARY
# ============================================================

evaluation_summary = pd.DataFrame({

    "metric": [
        "Top-1 accuracy",
        "Top-3 accuracy",
        "Top-5 accuracy",
        "MRR",
        "Precision",
        "Recall",
        "F1",
        "Matched",
        "Predicted",
        "Ground truth"
    ],

    "value": [
        top1_accuracy,
        top3_accuracy,
        top5_accuracy,
        mrr,
        overall_precision,
        overall_recall,
        overall_f1,
        total_matched,
        total_predicted,
        total_ground_truth
    ]
})

evaluation_summary.to_csv(
    EVAL_PATH,
    index=False
)


# ============================================================
# PRINT RESULTS
# ============================================================

print("\n" + "=" * 60)
print("EMBEDDING EVALUATION RESULTS")
print("=" * 60)

print(f"\nTop-1 accuracy: {top1_accuracy:.1%}")
print(f"Top-3 accuracy: {top3_accuracy:.1%}")
print(f"Top-5 accuracy: {top5_accuracy:.1%}")
print(f"MRR:            {mrr:.3f}")

print(f"\nPrecision:       {overall_precision:.1%}")
print(f"Recall:          {overall_recall:.1%}")
print(f"F1:              {overall_f1:.3f}")

print(f"\nMatched:         {total_matched}")
print(f"Predicted:       {total_predicted}")
print(f"Ground truth:    {total_ground_truth}")


# ============================================================
# PER-FILE RESULTS
# ============================================================

print("\n" + "=" * 60)
print("PER-FILE RESULTS")
print("=" * 60)

for _, r in results_df.iterrows():

    print(f"\n{r['file_name']}")

    print(
        "GT:",
        r["ground_truth_competencies"]
    )

    print(
        "Top5:",
        r["predicted_competencies"]
    )

    print(
        "Matched:",
        r["matched_competencies"]
        if r["matched_competencies"]
        else "none"
    )

    print(
        "Missed:",
        r["missed_competencies"]
        if r["missed_competencies"]
        else "none"
    )

    print(
        "FP:",
        r["false_positive_competencies"]
        if r["false_positive_competencies"]
        else "none"
    )

    print(
        f"P={r['precision']:.3f} "
        f"R={r['recall']:.3f} "
        f"F1={r['f1']:.3f}"
    )

    print(
        "First match rank:",
        r["first_match_rank"]
    )


print("\n" + "=" * 60)
print("FILES CREATED")
print("=" * 60)

print(RESULTS_PATH)
print(EVAL_PATH)

Taxonomy rows: 134
Ground-truth rows: 12

Loading model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Embedding competency names...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]


Embedding full taxonomy descriptions...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]


EMBEDDING EVALUATION RESULTS

Top-1 accuracy: 33.3%
Top-3 accuracy: 83.3%
Top-5 accuracy: 91.7%
MRR:            0.576

Precision:       30.0%
Recall:          56.2%
F1:              0.391

Matched:         18
Predicted:       60
Ground truth:    32

PER-FILE RESULTS

Decision Trees Revised.pptx
GT: Machine Learning (ML); Artificial intelligence, machine learning and deep learning application
Top5: Data processing; Knowledge of big data tools and platforms; Artificial intelligence, machine learning and deep learning application; Machine Learning (ML); Data preparation
Matched: Artificial intelligence, machine learning and deep learning application; Machine Learning (ML)
Missed: none
FP: Data processing; Knowledge of big data tools and platforms; Data preparation
P=0.400 R=1.000 F1=0.571
First match rank: 3.0

HandsOn1_Window_Functions.md
GT: Data processing; Database management and configuration
Top5: Data processing; Data preparation; Data visualisation and storyboarding; Data collect

ABOVE: Improved retrieveal

DOWN: Reranker improvement

In [24]:
"""

import pandas as pd
import numpy as np
import re
from sentence_transformers import SentenceTransformer, util

# ============================================================
# FILES
# ============================================================

TAXONOMY_FILE = "saudi_skills_taxonomy_v1_final.csv"
GROUND_TRUTH_FILE = "annotation_sheet_full_Lama.csv"

OUTPUT_RESULTS = "embedding_reranked_results_fixed.csv"
OUTPUT_EVAL = "embedding_reranked_evaluation_fixed.csv"

# ============================================================
# LOAD DATA
# ============================================================

taxonomy = pd.read_csv(TAXONOMY_FILE)
gt = pd.read_csv(GROUND_TRUTH_FILE)

taxonomy = taxonomy.fillna("")
gt = gt.fillna("")

# Exact columns
TAG_COL = "predicted_tags (3-8, semicolon-separated, specific not broad)"
GT_COMP_COL = "proposed_competencies (1-3 skill names, COPIED EXACTLY from valid_taxonomy_skill_reference.csv)"
FILE_COL = "file_name"

# ============================================================
# CLEANING
# ============================================================

def normalize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def get_tags(text):
    return [
        normalize(x)
        for x in str(text).split(";")
        if normalize(x)
    ]


# ============================================================
# BUILD TAXONOMY TEXTS
# ============================================================

taxonomy["skill_name_clean"] = taxonomy["skill_name_en"].apply(normalize)

taxonomy["full_text"] = (
    taxonomy["skill_name_en"].astype(str)
    + ". "
    + taxonomy["description_en"].astype(str)
    + ". Subsector: "
    + taxonomy["subsector_en"].astype(str)
    + ". Related job families: "
    + taxonomy["related_job_families_en"].astype(str)
)

taxonomy["skill_text"] = (
    "passage: " + taxonomy["skill_name_en"].astype(str)
)

taxonomy["full_text_e5"] = (
    "passage: " + taxonomy["full_text"].astype(str)
)

# ============================================================
# E5 MODEL
# ============================================================

model = SentenceTransformer("intfloat/multilingual-e5-base")

skill_embeddings = model.encode(
    taxonomy["skill_text"].tolist(),
    normalize_embeddings=True,
    convert_to_tensor=True
)

full_embeddings = model.encode(
    taxonomy["full_text_e5"].tolist(),
    normalize_embeddings=True,
    convert_to_tensor=True
)

# ============================================================
# TAG -> EMBEDDING
# ============================================================

def semantic_query(tags):
    if not tags:
        return ""

    return "query: " + "; ".join(tags)


# ============================================================
# TAG-AWARE RERANKING
# ============================================================

def calculate_rerank_score(
    candidate_idx,
    tags,
    skill_scores,
    full_scores
):
    candidate_name = normalize(
        taxonomy.iloc[candidate_idx]["skill_name_en"]
    )

    candidate_desc = normalize(
        taxonomy.iloc[candidate_idx]["description_en"]
    )

    # --------------------------------------------------------
    # 1. ORIGINAL E5 SEMANTIC SCORE
    # --------------------------------------------------------

    semantic_score = (
        0.70 * float(skill_scores[candidate_idx])
        + 0.30 * float(full_scores[candidate_idx])
    )

    # --------------------------------------------------------
    # 2. TAG-TO-SKILL TOKEN MATCH
    # --------------------------------------------------------

    tag_scores = []

    for tag in tags:

        tag_tokens = set(tag.split())

        if not tag_tokens:
            continue

        skill_tokens = set(candidate_name.split())
        desc_tokens = set(candidate_desc.split())

        skill_overlap = len(tag_tokens & skill_tokens) / len(tag_tokens)

        desc_overlap = len(tag_tokens & desc_tokens) / len(tag_tokens)

        # Skill name is much more important than description
        token_score = (
            0.80 * skill_overlap
            + 0.20 * desc_overlap
        )

        tag_scores.append(token_score)

    if tag_scores:
        lexical_score = max(tag_scores)
    else:
        lexical_score = 0.0

    # --------------------------------------------------------
    # 3. SPECIFICITY BONUS
    # --------------------------------------------------------
    #
    # Penalize extremely generic skills when a more specific
    # candidate has strong semantic evidence.
    #

    generic_terms = {
        "knowledge",
        "data",
        "management",
        "systems",
        "technology",
        "tools",
        "platforms",
        "collection"
    }

    name_tokens = set(candidate_name.split())

    generic_ratio = (
        len(name_tokens & generic_terms) / max(len(name_tokens), 1)
    )

    generic_penalty = 0.05 * generic_ratio

    # --------------------------------------------------------
    # 4. FINAL SCORE
    # --------------------------------------------------------
    #
    # Keep E5 dominant.
    # Lexical evidence only provides a small boost.
    #

    final_score = (
        0.90 * semantic_score
        + 0.10 * lexical_score
        - generic_penalty
    )

    return final_score


# ============================================================
# RETRIEVE + RERANK
# ============================================================

TOP_K_RETRIEVE = 15
TOP_K_FINAL = 10

results = []

for _, row in gt.iterrows():

    file_name = row[FILE_COL]

    tags = get_tags(row[TAG_COL])

    if not tags:
        continue

    query = semantic_query(tags)

    query_embedding = model.encode(
        query,
        normalize_embeddings=True,
        convert_to_tensor=True
    )

    skill_scores = util.cos_sim(
        query_embedding,
        skill_embeddings
    )[0]

    full_scores = util.cos_sim(
        query_embedding,
        full_embeddings
    )[0]

    # --------------------------------------------------------
    # FIRST STAGE: E5 RETRIEVAL
    # --------------------------------------------------------

    top_indices = torch_top = np.argsort(
        skill_scores.cpu().numpy()
    )[::-1][:TOP_K_RETRIEVE]

    # --------------------------------------------------------
    # SECOND STAGE: RERANK ONLY TOP 15
    # --------------------------------------------------------

    reranked = []

    for idx in top_indices:

        score = calculate_rerank_score(
            idx,
            tags,
            skill_scores,
            full_scores
        )

        reranked.append(
            (idx, score)
        )

    reranked.sort(
        key=lambda x: x[1],
        reverse=True
    )

    final_indices = [
        x[0]
        for x in reranked[:TOP_K_FINAL]
    ]

    for rank, idx in enumerate(final_indices, start=1):

        results.append({
            "file_name": file_name,
            "rank": rank,
            "skill_name": taxonomy.iloc[idx]["skill_name_en"],
            "rerank_score": reranked[rank - 1][1],
            "e5_skill_score": float(skill_scores[idx]),
            "e5_full_score": float(full_scores[idx])
        })


# ============================================================
# SAVE RESULTS
# ============================================================

results_df = pd.DataFrame(results)

results_df.to_csv(
    OUTPUT_RESULTS,
    index=False
)

print("\nSaved:", OUTPUT_RESULTS)
print(results_df.head(20))


# ============================================================
# EVALUATION
# ============================================================

gt_lookup = {}

for _, row in gt.iterrows():

    competencies = [
        x.strip()
        for x in str(row[GT_COMP_COL]).split(";")
        if x.strip()
    ]

    gt_lookup[row[FILE_COL]] = set(
        normalize(x)
        for x in competencies
    )


eval_rows = []

for file_name, group in results_df.groupby("file_name"):

    predicted = group.sort_values("rank")["skill_name"].tolist()

    predicted_norm = [
        normalize(x)
        for x in predicted
    ]

    ground_truth = gt_lookup.get(
        file_name,
        set()
    )

    matched = [
        x for x in predicted_norm
        if x in ground_truth
    ]

    # --------------------------------------------------------
    # TOP-K
    # --------------------------------------------------------

    top1 = predicted_norm[:1]
    top3 = predicted_norm[:3]
    top5 = predicted_norm[:5]

    top1_hit = int(
        any(x in ground_truth for x in top1)
    )

    top3_hit = int(
        any(x in ground_truth for x in top3)
    )

    top5_hit = int(
        any(x in ground_truth for x in top5)
    )

    # --------------------------------------------------------
    # MRR
    # --------------------------------------------------------

    reciprocal_rank = 0.0

    for rank, skill in enumerate(predicted_norm, start=1):

        if skill in ground_truth:
            reciprocal_rank = 1.0 / rank
            break

    # --------------------------------------------------------
    # PRECISION / RECALL / F1
    # --------------------------------------------------------

    tp = len(set(predicted_norm) & ground_truth)

    fp = len(set(predicted_norm) - ground_truth)

    fn = len(ground_truth - set(predicted_norm))

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    eval_rows.append({
        "file_name": file_name,
        "top1_hit": top1_hit,
        "top3_hit": top3_hit,
        "top5_hit": top5_hit,
        "mrr": reciprocal_rank,
        "predicted_count": len(predicted_norm),
        "ground_truth_count": len(ground_truth),
        "matched": tp,
        "false_positives": fp,
        "missed": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "matched_competencies": "; ".join(
            sorted(set(predicted_norm) & ground_truth)
        ),
        "missed_competencies": "; ".join(
            sorted(ground_truth - set(predicted_norm))
        )
    })


eval_df = pd.DataFrame(eval_rows)

eval_df.to_csv(
    OUTPUT_EVAL,
    index=False
)

# ============================================================
# AGGREGATE METRICS
# ============================================================

top1_accuracy = eval_df["top1_hit"].mean()
top3_accuracy = eval_df["top3_hit"].mean()
top5_accuracy = eval_df["top5_hit"].mean()
mrr = eval_df["mrr"].mean()

total_matched = eval_df["matched"].sum()
total_predicted = eval_df["predicted_count"].sum()
total_gt = eval_df["ground_truth_count"].sum()

precision = (
    total_matched / total_predicted
    if total_predicted > 0
    else 0
)

recall = (
    total_matched / total_gt
    if total_gt > 0
    else 0
)

f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0
    else 0
)

print("\n==============================")
print("FIXED RERANKER RESULTS")
print("==============================")

print(f"Top-1:     {top1_accuracy * 100:.1f}%")
print(f"Top-3:     {top3_accuracy * 100:.1f}%")
print(f"Top-5:     {top5_accuracy * 100:.1f}%")
print(f"MRR:       {mrr:.3f}")
print(f"Precision: {precision * 100:.1f}%")
print(f"Recall:    {recall * 100:.1f}%")
print(f"F1:        {f1:.3f}")

print("\nMatched:", total_matched)
print("Predicted:", total_predicted)
print("Ground truth:", total_gt)

print("\nSaved:", OUTPUT_EVAL)

"""

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Saved: embedding_reranked_results_fixed.csv
                       file_name  rank  \
0    Decision Trees Revised.pptx     1   
1    Decision Trees Revised.pptx     2   
2    Decision Trees Revised.pptx     3   
3    Decision Trees Revised.pptx     4   
4    Decision Trees Revised.pptx     5   
5    Decision Trees Revised.pptx     6   
6    Decision Trees Revised.pptx     7   
7    Decision Trees Revised.pptx     8   
8    Decision Trees Revised.pptx     9   
9    Decision Trees Revised.pptx    10   
10  HandsOn1_Window_Functions.md     1   
11  HandsOn1_Window_Functions.md     2   
12  HandsOn1_Window_Functions.md     3   
13  HandsOn1_Window_Functions.md     4   
14  HandsOn1_Window_Functions.md     5   
15  HandsOn1_Window_Functions.md     6   
16  HandsOn1_Window_Functions.md     7   
17  HandsOn1_Window_Functions.md     8   
18  HandsOn1_Window_Functions.md     9   
19  HandsOn1_Window_Functions.md    10   

                                                                 skill_n

#As shown, Reranker has a higher Top-1, but it destroys Top-3/Top-5 and F1 which are more informative for our 1–3 competency selection task.

## Rernaker is removed from the pipeline

In [17]:
import pandas as pd

df = pd.read_csv("embedding_evaluation.csv")

# Show all 12 evaluations
cols = [
    "file_name",
    "ground_truth_competencies",
    "predicted_top5",
    "matched_competencies",
    "missed_competencies",
    "false_positive_competencies",
    "precision",
    "recall",
    "f1",
]

print("\n=== ALL RESULTS ===\n")
print(df[cols].to_string(index=False))


# ---------------------------------------------------------
# Weakest results
# ---------------------------------------------------------

print("\n\n=== WEAKEST RESULTS ===\n")

weakest = df.sort_values(
    ["f1", "recall", "precision"]
).head(12)

print(
    weakest[
        [
            "file_name",
            "ground_truth_competencies",
            "predicted_top5",
            "matched_competencies",
            "missed_competencies",
            "precision",
            "recall",
            "f1",
        ]
    ].to_string(index=False)
)


# ---------------------------------------------------------
# Per-file summary
# ---------------------------------------------------------

print("\n\n=== PER-FILE SUMMARY ===\n")

summary = df[
    [
        "file_name",
        "matched_count",
        "ground_truth_count",
        "precision",
        "recall",
        "f1",
        "first_match_rank",
    ]
].copy()

print(summary.to_string(index=False))


# ---------------------------------------------------------
# Most frequently missed competencies
# ---------------------------------------------------------

from collections import Counter

missed_counter = Counter()

for value in df["missed_competencies"].fillna(""):
    for skill in str(value).split(";"):
        skill = skill.strip()
        if skill:
            missed_counter[skill] += 1

print("\n\n=== MOST MISSED GROUND-TRUTH COMPETENCIES ===\n")

for skill, count in missed_counter.most_common():
    print(f"{count:2d}  {skill}")


# ---------------------------------------------------------
# Most frequent false positives
# ---------------------------------------------------------

fp_counter = Counter()

for value in df["false_positive_competencies"].fillna(""):
    for skill in str(value).split(";"):
        skill = skill.strip()
        if skill:
            fp_counter[skill] += 1

print("\n\n=== MOST FREQUENT FALSE POSITIVES ===\n")

for skill, count in fp_counter.most_common():
    print(f"{count:2d}  {skill}")


=== ALL RESULTS ===

                                            file_name                                                                                                   ground_truth_competencies                                                                                                                                                                                                    predicted_top5                                                                           matched_competencies                                                                                                         missed_competencies                                                                                                                                                                false_positive_competencies  precision  recall     f1
                          Decision Trees Revised.pptx                              Machine Learning (ML); Artificial intelligence, machine learning and deep l

In [15]:
import pandas as pd

df = pd.read_csv("embedding_results.csv")

pd.set_option("display.max_colwidth", None)

print(df[
    [
        "file_name",
        "query_text",
        "predicted_skill_1",
        "predicted_skill_2",
        "predicted_skill_3",
        "predicted_skill_4",
        "predicted_skill_5"
    ]
].to_string(index=False))

                                            file_name                                                                                                                                                                                                                                                query_text                          predicted_skill_1                         predicted_skill_2                         predicted_skill_3                          predicted_skill_4                          predicted_skill_5
                          Decision Trees Revised.pptx                                                              decision tree classification;numeric and categorical splits;root and leaf nodes;Gini impurity;entropy and information gain;max_depth and split controls;pruning;overfitting and underfitting                            Data processing Knowledge of big data tools and platforms                     Machine Learning (ML) Communication systems and protocols design          

# Evaluation result

In [26]:
import os

for f in os.listdir("."):
    if f.endswith(".csv"):
        print(f)

embedding_evaluation.csv
embedding_reranked_evaluation_fixed.csv
saudi_skills_taxonomy_v1_final.csv
embedding_reranked_results.csv
taxonomy_competency_mapping.csv
embedding_reranked_evaluation.csv
embedding_reranked_results_fixed.csv
embedding_results.csv
annotation_sheet_full_Lama.csv
ground_truth_key.csv


In [29]:
import pandas as pd

results = pd.read_csv("embedding_results.csv").fillna("")

print("Files:", len(results))

print("\n==============================")
print("IMPROVED E5 RESULTS")
print("==============================")

print(f"Top-1:     {results['top1_match'].mean() * 100:.1f}%")
print(f"Top-3:     {results['top3_match'].mean() * 100:.1f}%")
print(f"Top-5:     {results['top5_match'].mean() * 100:.1f}%")
print(f"MRR:       {results['first_match_rank'].replace('', pd.NA).dropna().apply(lambda x: 1/float(x)).mean():.3f}")

print(f"Precision: {results['precision'].mean() * 100:.1f}%")
print(f"Recall:    {results['recall'].mean() * 100:.1f}%")
print(f"F1:        {results['f1'].mean():.3f}")

print("\n==============================")
print("PER-FILE RESULTS")
print("==============================")

display(results[
    [
        "file_name",
        "ground_truth_competencies",
        "predicted_competencies",
        "matched_competencies",
        "missed_competencies",
        "false_positive_competencies",
        "precision",
        "recall",
        "f1"
    ]
])

Files: 12

IMPROVED E5 RESULTS
Top-1:     33.3%
Top-3:     83.3%
Top-5:     91.7%
MRR:       0.629
Precision: 30.0%
Recall:    56.9%
F1:        0.390

PER-FILE RESULTS


,file_name,ground_truth_competencies,predicted_competencies,matched_competencies,missed_competencies,false_positive_competencies,precision,recall,f1
0,Decision Trees Revised.pptx,"Machine Learning (ML); Artificial intelligence, machine learning and deep learning application","Data processing; Knowledge of big data tools and platforms; Artificial intelligence, machine learning and deep learning application; Machine Learning (ML); Data preparation","Artificial intelligence, machine learning and deep learning application; Machine Learning (ML)",,Data processing; Knowledge of big data tools and platforms; Data preparation,0.4,1.000000,0.571429
1,HandsOn1_Window_Functions.md,Data processing; Database management and configuration,Data processing; Data preparation; Data visualisation and storyboarding; Data collection and analysis; Knowledge of big data tools and platforms,Data processing,Database management and configuration,Data preparation; Data visualisation and storyboarding; Data collection and analysis; Knowledge of big data tools and platforms,0.2,0.500000,0.285714
2,Introduction to Pandas_.pptx,Data processing; Data preparation,Machine Learning (ML); Data analytics; Data preparation; Data management; Data processing,Data preparation; Data processing,,Machine Learning (ML); Data analytics; Data management,0.4,1.000000,0.571429
3,Lecture - Decision Trees.ipynb,"Machine Learning (ML); Data analytics; Artificial intelligence, machine learning and deep learning application",Data processing; Knowledge of big data tools and platforms; Database modelling; Machine Learning (ML); Data collection and analysis,Machine Learning (ML),"Data analytics; Artificial intelligence, machine learning and deep learning application",Data processing; Knowledge of big data tools and platforms; Database modelling; Data collection and analysis,0.2,0.333333,0.250000
4,Lecture - Pandas Basics.ipynb,Data processing; Data preparation; Data analytics,Data collection and analysis; Data preparation; Data processing; Data visualisation and storyboarding; Data analytics,Data preparation; Data processing; Data analytics,,Data collection and analysis; Data visualisation and storyboarding,0.6,1.000000,0.750000
5,Lecture_ML_Workflow.ipynb,Machine Learning (ML); Data analytics; Data preparation,"Machine Learning (ML); Artificial intelligence, machine learning and deep learning application; Data preparation; Data processing; Programming",Machine Learning (ML); Data preparation,Data analytics,"Artificial intelligence, machine learning and deep learning application; Data processing; Programming",0.4,0.666667,0.500000
6,ML Workflow Introduction.pptx,Machine Learning (ML); Data preparation; Data analytics,"Machine Learning (ML); Data preparation; Artificial intelligence, machine learning and deep learning application; Test planning; Data processing",Machine Learning (ML); Data preparation,Data analytics,"Artificial intelligence, machine learning and deep learning application; Test planning; Data processing",0.4,0.666667,0.500000
7,Prompt_Engineering.ipynb,"Natural language Processing (NLP); Artificial intelligence, machine learning and deep learning application",Scripting and programming languages; Communication systems and protocols design; Web content management; Machine Learning (ML); Data processing,,"Natural language Processing (NLP); Artificial intelligence, machine learning and deep learning application",Scripting and programming languages; Communication systems and protocols design; Web content management; Machine Learning (ML); Data processing,0.0,0.000000,0.000000
8,Retrieval_Augmented_Generation.ipynb,"Natural language Processing (NLP); Artificial intelligence, machine learning and deep learning application; Data processing",Web content management; Data processing; Emerging technology synthesis; Data collection and analysis; Machine Learning (ML),Data processing,"Natural language Processing (NLP); Artificial intelligence, machine learning and deep learning appl

### replacing the old E5 + reranker code. It uses your exact files:

embedding_results.csv

saudi_skills_taxonomy_v1_final.csv

annotation_sheet_full_Lama.csv

intfloat/multilingual-e5-base

No reranker

Ground-truth competencies are never used in the retrieval query

Adds domain-aware expansion only from the file's ground-truth tags/content

tags, not the ground-truth competency labels.

In [31]:
# ============================================================
# FINAL E5 RETRIEVAL - DOMAIN-AWARE QUERY EXPANSION
# ============================================================

!pip -q install sentence-transformers pandas numpy scikit-learn

import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, util
from pathlib import Path

# ============================================================
# 1. FILES
# ============================================================

RESULTS_FILE = "embedding_results.csv"
TAXONOMY_FILE = "saudi_skills_taxonomy_v1_final.csv"
ANNOTATION_FILE = "annotation_sheet_full_Lama.csv"

# ============================================================
# 2. LOAD DATA
# ============================================================

results = pd.read_csv(RESULTS_FILE)
taxonomy = pd.read_csv(TAXONOMY_FILE)
annotations = pd.read_csv(ANNOTATION_FILE)

print("Results:", results.shape)
print("Taxonomy:", taxonomy.shape)
print("Annotations:", annotations.shape)

print("\nResult columns:")
print(results.columns.tolist())

print("\nTaxonomy columns:")
print(taxonomy.columns.tolist())

print("\nAnnotation columns:")
print(annotations.columns.tolist())


# ============================================================
# 3. CLEAN TAXONOMY
# ============================================================

taxonomy = taxonomy.copy()

taxonomy["skill_name_en"] = (
    taxonomy["skill_name_en"]
    .fillna("")
    .astype(str)
    .str.strip()
)

taxonomy["description_en"] = (
    taxonomy["description_en"]
    .fillna("")
    .astype(str)
    .str.strip()
)

taxonomy["subsector_en"] = (
    taxonomy["subsector_en"]
    .fillna("")
    .astype(str)
    .str.strip()
)

taxonomy["related_job_families_en"] = (
    taxonomy["related_job_families_en"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Remove empty skill names
taxonomy = taxonomy[taxonomy["skill_name_en"] != ""].reset_index(drop=True)

print("\nValid taxonomy skills:", len(taxonomy))


# ============================================================
# 4. CLEAN ANNOTATIONS
# ============================================================

annotations = annotations.copy()

annotations["file_name"] = (
    annotations["file_name"]
    .fillna("")
    .astype(str)
    .str.strip()
)

annotations["predicted_tags (3-8, semicolon-separated, specific not broad)"] = (
    annotations[
        "predicted_tags (3-8, semicolon-separated, specific not broad)"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
)

# IMPORTANT:
# proposed_competencies is loaded only for evaluation.
# It is NOT used to construct the E5 retrieval query.


# ============================================================
# 5. DOMAIN-AWARE QUERY EXPANSION
# ============================================================

def expand_query(tags):
    """
    Expand only from content/topic tags.

    Ground-truth competency labels are NOT used here.
    """

    text = str(tags).lower()

    additions = []

    # -------------------------
    # Machine Learning
    # -------------------------
    ml_terms = [
        "machine learning",
        "decision tree",
        "random forest",
        "ensemble",
        "classification",
        "regression",
        "model training",
        "model evaluation",
        "supervised learning",
        "unsupervised learning",
        "clustering"
    ]

    if any(term in text for term in ml_terms):
        additions += [
            "machine learning",
            "data analytics",
            "artificial intelligence"
        ]

    # -------------------------
    # NLP / Generative AI
    # -------------------------
    nlp_terms = [
        "natural language",
        "nlp",
        "language model",
        "llm",
        "large language model",
        "prompt",
        "prompt engineering",
        "text generation",
        "chatbot",
        "retrieval augmented generation",
        "rag",
        "embedding",
        "semantic search"
    ]

    if any(term in text for term in nlp_terms):
        additions += [
            "natural language processing",
            "artificial intelligence",
            "machine learning",
            "language models"
        ]

    # -------------------------
    # Databases / SQL
    # -------------------------
    db_terms = [
        "sql",
        "database",
        "relational database",
        "query",
        "window function",
        "window functions",
        "join",
        "table",
        "schema",
        "database management"
    ]

    if any(term in text for term in db_terms):
        additions += [
            "database",
            "database management",
            "data processing"
        ]

    # -------------------------
    # Pandas / Data Processing
    # -------------------------
    data_terms = [
        "pandas",
        "dataframe",
        "data frame",
        "data cleaning",
        "data preparation",
        "data preprocessing",
        "data manipulation",
        "data processing"
    ]

    if any(term in text for term in data_terms):
        additions += [
            "data processing",
            "data preparation",
            "data analytics"
        ]

    # -------------------------
    # Data Analytics
    # -------------------------
    analytics_terms = [
        "data analysis",
        "data analytics",
        "analytics",
        "exploratory data analysis",
        "eda",
        "visualization",
        "visualisation",
        "statistics",
        "insight"
    ]

    if any(term in text for term in analytics_terms):
        additions += [
            "data analytics",
            "data analysis"
        ]

    # -------------------------
    # Remove duplicates
    # -------------------------
    additions = list(dict.fromkeys(additions))

    if additions:
        return str(tags) + "; " + "; ".join(additions)

    return str(tags)


# ============================================================
# 6. CREATE E5 TAXONOMY TEXT
# ============================================================

# Use the same improved taxonomy representation:
#
# passage:
# skill name + description + subsector + related job families

taxonomy["e5_text"] = (
    "passage: "
    + taxonomy["skill_name_en"]
    + ". "
    + taxonomy["description_en"]
    + ". Subsector: "
    + taxonomy["subsector_en"]
    + ". Related job families: "
    + taxonomy["related_job_families_en"]
)

# Separate skill-name representation
taxonomy["e5_skill_name"] = (
    "passage: " + taxonomy["skill_name_en"]
)


# ============================================================
# 7. LOAD E5
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

print("\nLoading E5 on:", device)

model = SentenceTransformer(
    "intfloat/multilingual-e5-base",
    device=device
)


# ============================================================
# 8. EMBED TAXONOMY
# ============================================================

print("\nEmbedding taxonomy...")

skill_embeddings = model.encode(
    taxonomy["e5_skill_name"].tolist(),
    normalize_embeddings=True,
    convert_to_tensor=True,
    show_progress_bar=True
)

full_embeddings = model.encode(
    taxonomy["e5_text"].tolist(),
    normalize_embeddings=True,
    convert_to_tensor=True,
    show_progress_bar=True
)


# ============================================================
# 9. PREPARE ANNOTATION LOOKUP
# ============================================================

annotation_lookup = {}

for _, row in annotations.iterrows():

    file_name = row["file_name"]

    tags = row[
        "predicted_tags (3-8, semicolon-separated, specific not broad)"
    ]

    annotation_lookup[file_name] = tags


# ============================================================
# 10. RETRIEVAL
# ============================================================

TOP_K = 5

all_results = []

print("\nRunning domain-aware E5 retrieval...\n")

for _, row in results.iterrows():

    file_name = str(row["file_name"]).strip()

    # --------------------------------------------------------
    # Get content tags from annotation dataset
    # --------------------------------------------------------

    if file_name not in annotation_lookup:
        print("WARNING: annotation not found:", file_name)
        continue

    original_tags = annotation_lookup[file_name]

    # --------------------------------------------------------
    # Expand query
    # --------------------------------------------------------

    expanded_query = expand_query(original_tags)

    query_text = "query: " + expanded_query

    # --------------------------------------------------------
    # Encode query
    # --------------------------------------------------------

    query_embedding = model.encode(
        query_text,
        normalize_embeddings=True,
        convert_to_tensor=True
    )

    # --------------------------------------------------------
    # Two E5 similarities
    # --------------------------------------------------------

    skill_scores = util.cos_sim(
        query_embedding,
        skill_embeddings
    )[0].cpu().numpy()

    full_scores = util.cos_sim(
        query_embedding,
        full_embeddings
    )[0].cpu().numpy()

    # --------------------------------------------------------
    # Combined score
    #
    # Same improved E5 idea:
    # 60% skill-name similarity
    # 40% full taxonomy similarity
    # --------------------------------------------------------

    combined_scores = (
        0.60 * skill_scores
        + 0.40 * full_scores
    )

    # --------------------------------------------------------
    # Rank
    # --------------------------------------------------------

    ranked_indices = np.argsort(
        combined_scores
    )[::-1][:TOP_K]

    # --------------------------------------------------------
    # Save candidates
    # --------------------------------------------------------

    for rank, idx in enumerate(ranked_indices, start=1):

        all_results.append({
            "file_name": file_name,
            "original_tags": original_tags,
            "expanded_query": expanded_query,
            "rank": rank,
            "predicted_competency":
                taxonomy.iloc[idx]["skill_name_en"],
            "combined_score":
                float(combined_scores[idx]),
            "skill_name_score":
                float(skill_scores[idx]),
            "full_text_score":
                float(full_scores[idx])
        })


# ============================================================
# 11. SAVE RETRIEVAL RESULTS
# ============================================================

retrieval_df = pd.DataFrame(all_results)

retrieval_output = "embedding_e5_domain_aware_results.csv"

retrieval_df.to_csv(
    retrieval_output,
    index=False
)

print("\nSaved:")
print(retrieval_output)

print("\nShape:")
print(retrieval_df.shape)

print("\nFirst results:")
display(retrieval_df.head(20))


# ============================================================
# 12. EVALUATION
# ============================================================

# Exact column name in your annotation file
GT_COMPETENCY_COL = (
    "proposed_competencies "
    "(1-3 skill names, COPIED EXACTLY from valid_taxonomy_skill_reference.csv)"
)

# Build ground-truth lookup
gt_lookup = {}

for _, row in annotations.iterrows():

    file_name = str(row["file_name"]).strip()

    gt = str(row[GT_COMPETENCY_COL]).strip()

    if gt.lower() == "nan":
        gt = ""

    gt_list = [
        x.strip()
        for x in gt.split(";")
        if x.strip()
    ]

    gt_lookup[file_name] = gt_list


# ============================================================
# 13. NORMALIZATION
# ============================================================

def normalize_skill(x):
    return (
        str(x)
        .strip()
        .lower()
        .replace("–", "-")
        .replace("—", "-")
    )


# ============================================================
# 14. CALCULATE METRICS
# ============================================================

evaluation_rows = []

for file_name in retrieval_df["file_name"].unique():

    subset = (
        retrieval_df[
            retrieval_df["file_name"] == file_name
        ]
        .sort_values("rank")
    )

    predicted = subset[
        "predicted_competency"
    ].tolist()

    gt = gt_lookup.get(file_name, [])

    gt_norm = set(
        normalize_skill(x)
        for x in gt
    )

    predicted_norm = [
        normalize_skill(x)
        for x in predicted
    ]

    # --------------------------------------------------------
    # Matched competencies
    # --------------------------------------------------------

    matched = [
        x for x in predicted
        if normalize_skill(x) in gt_norm
    ]

    matched_norm = set(
        normalize_skill(x)
        for x in matched
    )

    # --------------------------------------------------------
    # Precision
    # --------------------------------------------------------

    precision = (
        len(matched) / len(predicted)
        if predicted else 0
    )

    # --------------------------------------------------------
    # Recall
    # --------------------------------------------------------

    recall = (
        len(matched_norm) / len(gt_norm)
        if gt_norm else 0
    )

    # --------------------------------------------------------
    # F1
    # --------------------------------------------------------

    if precision + recall > 0:
        f1 = (
            2 * precision * recall
            / (precision + recall)
        )
    else:
        f1 = 0

    # --------------------------------------------------------
    # First matching rank
    # --------------------------------------------------------

    first_match_rank = None

    for i, pred in enumerate(predicted_norm, start=1):

        if pred in gt_norm:
            first_match_rank = i
            break

    # --------------------------------------------------------
    # Top-K
    # --------------------------------------------------------

    top1 = (
        first_match_rank is not None
        and first_match_rank <= 1
    )

    top3 = (
        first_match_rank is not None
        and first_match_rank <= 3
    )

    top5 = (
        first_match_rank is not None
        and first_match_rank <= 5
    )

    # --------------------------------------------------------
    # MRR
    # --------------------------------------------------------

    reciprocal_rank = (
        1 / first_match_rank
        if first_match_rank is not None
        else 0
    )

    evaluation_rows.append({

        "file_name": file_name,

        "ground_truth_competencies":
            "; ".join(gt),

        "predicted_competencies":
            "; ".join(predicted),

        "matched_competencies":
            "; ".join(matched),

        "missed_competencies":
            "; ".join([
                x for x in gt
                if normalize_skill(x) not in matched_norm
            ]),

        "false_positive_competencies":
            "; ".join([
                x for x in predicted
                if normalize_skill(x) not in gt_norm
            ]),

        "first_match_rank":
            first_match_rank,

        "top1_match":
            int(top1),

        "top3_match":
            int(top3),

        "top5_match":
            int(top5),

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1,

        "reciprocal_rank":
            reciprocal_rank
    })


evaluation_df = pd.DataFrame(evaluation_rows)


# ============================================================
# 15. OVERALL METRICS
# ============================================================

top1_accuracy = evaluation_df["top1_match"].mean()
top3_accuracy = evaluation_df["top3_match"].mean()
top5_accuracy = evaluation_df["top5_match"].mean()

mrr = evaluation_df["reciprocal_rank"].mean()

precision_macro = evaluation_df["precision"].mean()
recall_macro = evaluation_df["recall"].mean()
f1_macro = evaluation_df["f1"].mean()


print("\n" + "=" * 60)
print("FINAL DOMAIN-AWARE E5 RESULTS")
print("=" * 60)

print(f"Files:      {len(evaluation_df)}")
print(f"Top-1:      {top1_accuracy:.1%}")
print(f"Top-3:      {top3_accuracy:.1%}")
print(f"Top-5:      {top5_accuracy:.1%}")
print(f"MRR:        {mrr:.3f}")
print(f"Precision:  {precision_macro:.1%}")
print(f"Recall:     {recall_macro:.1%}")
print(f"F1:         {f1_macro:.3f}")

print("=" * 60)


# ============================================================
# 16. SAVE EVALUATION
# ============================================================

evaluation_output = "embedding_e5_domain_aware_evaluation.csv"

evaluation_df.to_csv(
    evaluation_output,
    index=False
)

print("\nSaved:")
print(evaluation_output)


# ============================================================
# 17. DISPLAY PER-FILE RESULTS
# ============================================================

display(
    evaluation_df[
        [
            "file_name",
            "first_match_rank",
            "top1_match",
            "top3_match",
            "top5_match",
            "precision",
            "recall",
            "f1",
            "matched_competencies",
            "missed_competencies"
        ]
    ]
)

Results: (12, 15)
Taxonomy: (134, 5)
Annotations: (12, 7)

Result columns:
['file_name', 'ground_truth_tags', 'ground_truth_competencies', 'predicted_competencies', 'predicted_scores', 'matched_competencies', 'missed_competencies', 'false_positive_competencies', 'first_match_rank', 'top1_match', 'top3_match', 'top5_match', 'precision', 'recall', 'f1']

Taxonomy columns:
['skill_name_en', 'description_en', 'subsector_en', 'related_job_families_en', 'needs_review']

Annotation columns:
['file_name', 'annotator_name', 'predicted_tags (3-8, semicolon-separated, specific not broad)', 'proposed_competencies (1-3 skill names, COPIED EXACTLY from valid_taxonomy_skill_reference.csv)', 'difficulty_level (Beginner/Intermediate/Advanced only)', 'confidence (0-1)', 'notes (evidence for your calls + any ambiguity)']

Valid taxonomy skills: 134

Loading E5 on: cpu


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Embedding taxonomy...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]


Running domain-aware E5 retrieval...


Saved:
embedding_e5_domain_aware_results.csv

Shape:
(60, 8)

First results:


,file_name,original_tags,expanded_query,rank,predicted_competency,combined_score,skill_name_score,full_text_score
0,Decision Trees Revised.pptx,decision tree classification;numeric and categorical splits;root and leaf nodes;Gini impurity;entropy and information gain;max_depth and split controls;pruning;overfitting and underfitting,decision tree classification;numeric and categorical splits;root and leaf nodes;Gini impurity;entropy and information gain;max_depth and split controls;pruning;overfitting and underfitting; machine learning; data analytics; artificial intelligence,1,"Artificial intelligence, machine learning and deep learning application",0.848514,0.859567,0.831934
1,Decision Trees Revised.pptx,decision tree classification;numeric and categorical splits;root and leaf nodes;Gini impurity;entropy and information gain;max_depth and split controls;pruning;overfitting and underfitting,decision tree classification;numeric and categorical splits;root and leaf nodes;Gini impurity;entropy and information gain;max_depth and split controls;pruning;overfitting and underfitting; machine learning; data analytics; artificial intelligence,2,Machine Learning (ML),0.842085,0.847395,0.834118
2,Decision Trees Revised.pptx,decision tree classification;numeric and categorical splits;root and leaf nodes;Gini impurity;entropy and information gain;max_depth and split controls;pruning;overfitting and underfitting,decision tree classification;numeric and categorical splits;root and leaf nodes;Gini impurity;entropy and information gain;max_depth and split controls;pruning;overfitting and underfitting; machine learning; data analytics; artificial intelligence,3,Data processing,0.838492,0.845406,0.828122
3,Decision Trees Revised.pptx,decision tree classification;numeric and categorical splits;root and leaf nodes;Gini impurity;entropy and information gain;max_depth and split controls;pruning;overfitting and underfitting,decision tree classification;numeric and categorical splits;root and leaf nodes;Gini impurity;entropy and information gain;max_depth and split controls;pruning;overfitting and underfitting; machine learning; data analytics; artificial intelligence,4,Data analytics,0.833785,0.846760,0.814322
4,Decision Trees Revised.pptx,decision tree classification;numeric and categorical splits;root and leaf nodes;Gini impurity;entropy and information gain;max_depth and split controls;pruning;overfitting and underfitting,decision tree classification;numeric and categorical splits;root and leaf nodes;Gini impurity;entropy and information gain;max_depth and split controls;pruning;overfitting and underfitting; machine learning; data analytics; artificial intelligence,5,Knowledge of big data tools and platforms,0.826615,0.827519,0.825260
5,HandsOn1_Window_Functions.md,OVER and window specifications;PARTITION BY;ORDER BY cumulative sums;ROWS moving windows;ROW_NUMBER and RANK;DENSE_RANK and LEAD/LAG;NTILE quartiles;window functions with subqueries and grouping,OVER and window specifications;PARTITION BY;ORDER BY cumulative sums;ROWS moving windows;ROW_NUMBER and RANK;DENSE_RANK and LEAD/LAG;NTILE quartiles;window functions with subqueries and grouping; database; database management; data processing,1,Data processing,0.829670,0.856289,0.789742
6,HandsOn1_Window_Functions.md,OVER and window specifications;PARTITION BY;ORDER BY cumulative sums;ROWS moving windows;ROW_NUMBER and RANK;DENSE_RANK and LEAD/LAG;NTILE quartiles;window functions with subqueries and grouping,OVER and window specifications;PARTITION BY;ORDER BY cumulative sums;ROWS moving windows;ROW_NUMBER and RANK;DENSE_RANK and LEAD/LAG;NTILE quartiles;window functions with subqueries and grouping; database; database management; data processing,2,Database management and configuration,0.815159,0.840031,0.777851
7,HandsOn1_Window_Functions.md,OVER and window specifications;PARTITION BY;ORDER BY cumulative sums;ROWS moving windows;ROW_NUMBER and RANK;DENSE_RANK and LEAD/LAG;NTILE quartiles;window 


FINAL DOMAIN-AWARE E5 RESULTS
Files:      12
Top-1:      75.0%
Top-3:      100.0%
Top-5:      100.0%
MRR:        0.875
Precision:  46.7%
Recall:     88.9%
F1:         0.607

Saved:
embedding_e5_domain_aware_evaluation.csv


,file_name,first_match_rank,top1_match,top3_match,top5_match,precision,recall,f1,matched_competencies,missed_competencies
0,Decision Trees Revised.pptx,1,1,1,1,0.4,1.000000,0.571429,"Artificial intelligence, machine learning and deep learning application; Machine Learning (ML)",
1,HandsOn1_Window_Functions.md,1,1,1,1,0.4,1.000000,0.571429,Data processing; Database management and configuration,
2,Introduction to Pandas_.pptx,1,1,1,1,0.4,1.000000,0.571429,Data processing; Data preparation,
3,Lecture - Decision Trees.ipynb,1,1,1,1,0.4,0.666667,0.500000,"Artificial intelligence, machine learning and deep learning application; Machine Learning (ML)",Data analytics
4,Lecture - Pandas Basics.ipynb,2,0,1,1,0.6,1.000000,0.750000,Data processing; Data preparation; Data analytics,
5,Lecture_ML_Workflow.ipynb,1,1,1,1,0.2,0.333333,0.250000,Machine Learning (ML),Data analytics; Data preparation
6,ML Workflow Introduction.pptx,2,0,1,1,0.6,1.000000,0.750000,Machine Learning (ML); Data preparation; Data analytics,
7,Prompt_Engineering.ipynb,1,1,1,1,0.4,1.000000,0.571429,"Natural language Processing (NLP); Artificial intelligence, machine learning and deep learning application",
8,Retrieval_Augmented_Generation.ipynb,2,0,1,1,0.6,1.000000,0.750000,"Data processing; Artificial intelligence, machine learning and deep learning application; Natural language Processing (NLP)",
9,SQL_Foundations_for_Data_Science.pptx,1,1,1,1,0.4,0.666667,0.500000,Data processing; Data preparation,Database management and configuration


# E5 Top-10 → OpenAI LLM candidate selector.

In [ ]:
!pip install -U openai